# ⚙️ Notebook 3: Feature Engineering

Transform raw features into ML-ready inputs: encoding, scaling, selection.

In [ ]:
import pandas as pd, numpy as np
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.feature_selection import SelectKBest, chi2, mutual_info_classif
import matplotlib.pyplot as plt, seaborn as sns
import warnings; warnings.filterwarnings('ignore')

df = pd.read_csv('../data/processed/cleaned_churn_data.csv')
print(f"Shape: {df.shape}")

## 3.1 Binary Encoding

In [ ]:
binary_cols = ['Partner','Dependents','PhoneService','PaperlessBilling']
for col in binary_cols:
    df[col] = df[col].map({'Yes':1,'No':0})
df['gender'] = df['gender'].map({'Male':1,'Female':0})
print("Binary encoding done")
df[binary_cols + ['gender']].head()

## 3.2 One-Hot Encoding

In [ ]:
cat_cols = ['InternetService','Contract','PaymentMethod',
            'MultipleLines','OnlineSecurity','OnlineBackup',
            'DeviceProtection','TechSupport','StreamingTV','StreamingMovies']
df_enc = pd.get_dummies(df, columns=cat_cols, drop_first=False)
df_enc.fillna(0, inplace=True)
print(f"After one-hot: {df_enc.shape[1]} columns")

## 3.3 Feature Scaling

In [ ]:
numeric = ['tenure','MonthlyCharges','TotalCharges','SeniorCitizen']
scaler = StandardScaler()

X = df_enc.drop(columns=['Churn','customerID','TenureCohort'], errors='ignore')
y = df_enc['Churn']

X[numeric] = scaler.fit_transform(X[numeric])
print(f"Features scaled: {numeric}")
X[numeric].describe().round(3)

## 3.4 Feature Importance — Mutual Information

In [ ]:
from sklearn.feature_selection import mutual_info_classif
mi = mutual_info_classif(X.fillna(0), y, random_state=42)
mi_series = pd.Series(mi, index=X.columns).sort_values(ascending=False)

print("Top 15 features by Mutual Information:")
print(mi_series.head(15).round(4).to_string())

mi_series.head(15)[::-1].plot.barh(figsize=(9,6), color='#2dd4bf')
plt.title('Feature Importance — Mutual Information', fontweight='bold')
plt.xlabel('MI Score'); plt.tight_layout(); plt.show()

## 3.5 Correlation with Target

In [ ]:
corr = X.corrwith(y).abs().sort_values(ascending=False)
print("Top 15 features correlated with Churn:")
print(corr.head(15).round(4).to_string())

## 3.6 Save Engineered Dataset

In [ ]:
X['Churn'] = y.values
X.to_csv('../data/processed/engineered_features.csv', index=False)
print(f"Saved engineered dataset: {X.shape}")